# 📖 Notebook 2: Database Replication & Failover

## Why This Matters

In 2017, GitLab accidentally deleted a production database. They had backups,
but the restore took **18 hours**. During that time, they lost 6 hours of data.

With **streaming replication**, a standby database receives every change in
real-time. If the primary crashes, you promote the standby — and you are back
online in seconds, not hours.

This is how banks, stock exchanges, and cloud providers keep databases running 24/7.

## Learning Objectives

- Understand how PostgreSQL streaming replication works
- Monitor replication lag and WAL (Write-Ahead Log) status
- Perform a manual failover (promote standby to primary)
- Understand the split-brain problem and how to prevent it

## 🛠️ Setup

Make sure all services are running:

```bash
cd 08-enterprise/bcdr
docker compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).

In [ ]:
import psycopg2
import subprocess
import time
from tabulate import tabulate

DB_PRIMARY = {
    "host": "localhost", "port": 5432,
    "database": "bcdr_demo", "user": "demo", "password": "demo"
}
DB_STANDBY = {
    "host": "localhost", "port": 55433,
    "database": "bcdr_demo", "user": "demo", "password": "demo"
}

def get_primary_connection():
    return psycopg2.connect(**DB_PRIMARY)

def get_standby_connection():
    return psycopg2.connect(**DB_STANDBY)

def docker_exec(container, cmd, user=None):
    """Run a command inside a Docker container.

    `docker exec` runs as root unless told otherwise, and some PostgreSQL
    binaries refuse that outright -- `pg_ctl` exits with "cannot be run as
    root" because the server must never own files as root. Client tools
    (psql, pg_dump, pg_restore) do not care. Pass user="postgres" for the
    server-side ones."""
    prefix = ["docker", "exec"] + (["-u", user] if user else [])
    result = subprocess.run(
        prefix + [container] + cmd,
        capture_output=True, text=True, timeout=30
    )
    return result.stdout.strip(), result.stderr.strip()

# Test connections
for name, cfg in [('Primary', DB_PRIMARY), ('Standby', DB_STANDBY)]:
    try:
        conn = psycopg2.connect(**cfg)
        conn.close()
        print(f"✅ {name} connected (port {cfg['port']})")
    except Exception as e:
        print(f"❌ {name} failed: {e}")

## 📚 How Streaming Replication Works

PostgreSQL streaming replication uses the **Write-Ahead Log (WAL)**:

```
1. Client sends a write (INSERT/UPDATE/DELETE) to the primary
2. Primary writes the change to the WAL file (on disk)
3. Primary sends the WAL records to all connected standbys
4. Standby receives WAL records and replays them
5. Standby now has the same data as the primary
```

### WAL (Write-Ahead Log)

The WAL is a sequential log of every change made to the database.
Think of it like a bank transaction log — before money moves,
the transaction is written to the log first.

This is the foundation of both replication AND backup recovery.

### Asynchronous vs Synchronous Replication

| Mode | How It Works | Trade-off |
|------|-------------|-----------|
| **Async** (default) | Primary does NOT wait for standby to confirm | Faster writes, but standby may be slightly behind |
| **Sync** | Primary waits for standby to confirm EACH write | Slower writes, but zero data loss (RPO = 0) |

In [ ]:
# =============================================================================
# Demo: Inspect WAL Status on Primary
# =============================================================================

conn = get_primary_connection()
cur = conn.cursor()

# Current WAL position on primary
cur.execute("SELECT pg_current_wal_lsn(), pg_walfile_name(pg_current_wal_lsn())")
lsn, wal_file = cur.fetchone()

print("=" * 65)
print("WAL STATUS ON PRIMARY")
print("=" * 65)
print(f"  Current WAL LSN:  {lsn}")
print(f"  Current WAL file: {wal_file}")
print()
print("LSN = Log Sequence Number — a pointer to a position in the WAL.")
print("Each write advances the LSN. The standby tries to keep up.")

# Check replication slots
cur.execute(
    "SELECT slot_name, active, restart_lsn, confirmed_flush_lsn "
    "FROM pg_replication_slots"
)
slots = cur.fetchall()
print()
print("Replication Slots:")
for s in slots:
    print(f"  Slot: {s[0]}, Active: {s[1]}, Restart LSN: {s[2]}")
print()
print("💡 Replication slots prevent the primary from discarding WAL files")
print("   that the standby has not yet received.")

conn.close()

In [ ]:
# =============================================================================
# Demo: Monitor Replication Lag
# =============================================================================

conn = get_primary_connection()
cur = conn.cursor()

cur.execute(
    "SELECT pid, client_addr, state, "
    "pg_wal_lsn_diff(sent_lsn, write_lsn) as send_lag_bytes, "
    "pg_wal_lsn_diff(sent_lsn, replay_lsn) as replay_lag_bytes, "
    "sync_state "
    "FROM pg_stat_replication"
)
rows = cur.fetchall()
conn.close()

print("=" * 65)
print("REPLICATION LAG MONITOR")
print("=" * 65)

if rows:
    table = []
    for r in rows:
        table.append([r[0], r[1], r[2], f"{r[3]} bytes", f"{r[4]} bytes", r[5]])
    print(tabulate(table,
        headers=["PID", "Address", "State", "Send Lag", "Replay Lag", "Sync"],
        tablefmt="grid"))
    print()
    print("💡 Send Lag = bytes sent but not yet written by standby")
    print("   Replay Lag = bytes sent but not yet replayed by standby")
    print("   Both should be 0 or very small during normal operation.")
else:
    print("⚠️  No standbys connected")

## 📚 Verifying Replication Works

The best way to verify replication is simple:
1. Write data to the primary
2. Read it from the standby
3. Confirm they match

The standby is **read-only** — you cannot write to it. This is a safety feature.

In [ ]:
# =============================================================================
# Demo: Write to Primary, Read from Standby
# =============================================================================

# Write to primary
primary_conn = get_primary_connection()
primary_conn.autocommit = True
primary_cur = primary_conn.cursor()

marker = f"REPL_TEST_{int(time.time())}"
primary_cur.execute(
    "INSERT INTO audit_log (table_name, record_id, action, changed_by) "
    "VALUES (%s, %s, %s, %s) RETURNING id",
    ('repl_test', 999, 'INSERT', marker)
)
new_id = primary_cur.fetchone()[0]
print(f"✅ Wrote record #{new_id} to PRIMARY with marker: {marker}")

# Poll the standby instead of sleeping a magic number. A fixed sleep either
# wastes time or flakes on a slower machine -- and it hides the lag.
standby_conn = get_standby_connection()
standby_cur = standby_conn.cursor()

deadline = time.time() + 10
row = None
while time.time() < deadline:
    standby_conn.rollback()  # end the snapshot so we can see new commits
    standby_cur.execute(
        "SELECT id, table_name, action, changed_by FROM audit_log WHERE id = %s",
        (new_id,)
    )
    row = standby_cur.fetchone()
    if row:
        break
    time.sleep(0.01)

assert row is not None, (
    f"record #{new_id} never reached the standby -- streaming replication is "
    f"not working, so the failover later in this notebook would silently lose "
    f"everything written since the stream broke")
print(f"✅ Read record #{row[0]} from STANDBY: {row[3]}")
print("\n🎉 Replication is working! Data written to primary appears on standby.")

# Verify standby is read-only
rejected = False
try:
    standby_cur.execute(
        "INSERT INTO audit_log (table_name, record_id, action) "
        "VALUES ('test', 0, 'TEST')"
    )
except psycopg2.Error as e:
    rejected = True
    print(f"\n✅ Standby correctly rejected write: {type(e).__name__}")
    print("   Hot standbys are READ-ONLY. This prevents split-brain.")
finally:
    standby_conn.rollback()

assert rejected, (
    "the standby ACCEPTED a write -- a writable standby means both nodes can "
    "diverge under a partition, which is exactly the split-brain failure the "
    "next section warns about")

# Cleanup
primary_cur.execute("DELETE FROM audit_log WHERE changed_by = %s", (marker,))
primary_conn.close()
standby_conn.close()


## 📚 The Split-Brain Problem

**Split-brain** is the most dangerous scenario in database failover:

```
  [Primary A] ←── network partition ──→ [Primary B]
       ↓                                      ↓
  accepts writes                        accepts writes
  (different data!)                     (different data!)
```

Both servers think they are the primary and accept writes independently.
The data **diverges** — and merging it back together is extremely difficult.

### Prevention Strategies

| Strategy | How It Works |
|----------|-------------|
| **Fencing (STONITH)** | Physically shut down the old primary before promoting standby |
| **Quorum** | Require majority agreement (odd number of nodes: 3, 5, 7) |
| **Watchdog Timer** | Node auto-shuts-down if it loses contact with the cluster |
| **Lease-based** | Primary must periodically renew a lease; if it cannot, standby takes over |

In this lab, we will do **manual failover with explicit fencing** (stopping the primary first).

## 📚 Manual Failover Procedure

Here is the step-by-step procedure to failover from primary to standby:

```
Step 1: RECORD where the primary's WAL ends (the data the standby must have)
Step 2: FENCE the primary (stop it from accepting writes)
Step 3: VERIFY the standby received everything up to that point
Step 4: PROMOTE the standby to become the new primary
Step 5: VERIFY the new primary is accepting writes
Step 6: FAILBACK — rebuild the fenced node and re-form the pair
```

**Important**: In a real production system, you would use tools like
**Patroni**, **pg_auto_failover**, or **Pacemaker** to automate steps 1-5.
We do them manually here so you understand what happens under the hood.

**Step 6 is the half everyone skips.** A cluster that has failed over has no
standby left — it survived one failure and is now defenceless against the
next. Worse, the promoted node holds writes the fenced node never saw, so
bringing the fenced node back carelessly gives you two primaries with
diverged data. This notebook deliberately stops after step 5, in the messy
state a real on-call engineer is handed; notebooks 3 and 4 open with a
failback cell that does step 6 properly, including carrying those writes
back.


In [ ]:
# =============================================================================
# Demo: Check Standby Status Before Failover
# =============================================================================
# Before promoting, always check the standby is healthy and caught up.

standby_conn = get_standby_connection()
standby_cur = standby_conn.cursor()

# Is this server in recovery mode? (True = standby)
standby_cur.execute("SELECT pg_is_in_recovery()")
is_standby = standby_cur.fetchone()[0]

# Last WAL position received and replayed
standby_cur.execute(
    "SELECT pg_last_wal_receive_lsn(), "
    "pg_last_wal_replay_lsn(), "
    "pg_last_xact_replay_timestamp()"
)
received_lsn, replayed_lsn, last_replay_time = standby_cur.fetchone()
standby_conn.close()

print("=" * 65)
print("STANDBY STATUS CHECK")
print("=" * 65)
print(f"  Is in recovery (standby) mode: {is_standby}")
print(f"  Last WAL received:  {received_lsn}")
print(f"  Last WAL replayed:  {replayed_lsn}")
print(f"  Last replay time:   {last_replay_time}")

assert is_standby, (
    "port 55433 is NOT in recovery mode -- it is already a primary. Promoting "
    "it again is a no-op, and running the failover below would leave you with "
    "two primaries. Run the failback cell at the top of notebook 3 or 4 first.")
print("\n✅ Server is in standby mode — ready for promotion")
if received_lsn == replayed_lsn:
    print("✅ Standby is fully caught up (received == replayed)")
else:
    print("⚠️  Standby has unreplayed WAL — promotion will replay it first,")
    print("   but any WAL it never *received* is data you are about to lose.")

In [ ]:
# =============================================================================
# Demo: Perform Manual Failover
# =============================================================================
# WARNING: this changes your cluster topology on purpose, and LEAVES it
# changed. When this cell finishes, the primary is fenced (dead) and the
# standby is the new primary -- exactly the state step 6 of the procedure
# above exists to clean up. Notebooks 3 and 4 open with a failback cell that
# does that, and it carries the write we make below on the new primary with
# it. A failback that quietly drops the failover-window writes is data loss
# wearing a success message.

print('=' * 65)
print('MANUAL FAILOVER PROCEDURE')
print('=' * 65)

# --------------------------------------------------------------- Step 1 ----
# Record where the primary's WAL ends. Everything up to this LSN is data the
# standby MUST already hold if we are to fail over without losing anything.
primary_conn = get_primary_connection()
primary_cur = primary_conn.cursor()
primary_cur.execute("SELECT pg_current_wal_lsn()")
pre_lsn = primary_cur.fetchone()[0]
primary_conn.close()
print(f"\nStep 1: RECORD — primary WAL ends at {pre_lsn}")

# --------------------------------------------------------------- Step 2 ----
# FENCE. Stop the old primary BEFORE promoting anything. If both nodes were
# ever live at the same time they would accept conflicting writes: split-brain.
print("\nStep 2: FENCE — stopping the primary container (STONITH)...")
disaster_time = time.time()
subprocess.run(
    ["docker", "stop", "-t", "30", "bcdr-pg-primary"],
    capture_output=True, timeout=60
)
try:
    psycopg2.connect(connect_timeout=3, **DB_PRIMARY).close()
    raise AssertionError(
        "the primary is still accepting connections after the fence -- "
        "promoting now would create a genuine split-brain")
except psycopg2.Error:
    print("  ✅ Primary is fenced (connections refused).")

# --------------------------------------------------------------- Step 3 ----
# VERIFY the standby really received everything the primary had. The gap
# between pre_lsn and the standby's replay position IS the data loss of this
# failover -- the RPO, measured in bytes of WAL rather than guessed at.
standby_conn = get_standby_connection()
standby_cur = standby_conn.cursor()
standby_cur.execute(
    "SELECT pg_last_wal_replay_lsn(), "
    "GREATEST(pg_wal_lsn_diff(%s, pg_last_wal_replay_lsn()), 0)",
    (pre_lsn,)
)
replay_lsn, missing_bytes = standby_cur.fetchone()
standby_conn.close()
missing_bytes = int(missing_bytes)
print(f"\nStep 3: VERIFY — standby replayed up to {replay_lsn}")
print(f"  WAL the standby never replayed: {missing_bytes} bytes  <- this is the RPO")

# A graceful fence lets the primary flush and ship its remaining WAL before it
# exits, so a clean stop should cost us nothing. If this fires, the fence
# raced the replication stream and the failover is NOT lossless.
assert missing_bytes == 0, (
    f"{missing_bytes} bytes of WAL never reached the standby -- promoting now "
    f"loses those transactions. In production you would wait for the standby "
    f"to catch up, or accept the loss knowingly.")

# --------------------------------------------------------------- Step 4 ----
print("\nStep 4: PROMOTE — telling the standby it is now the primary...")
out, err = docker_exec(
    "bcdr-pg-standby",
    ["pg_ctl", "promote", "-D", "/var/lib/postgresql/data"],
    user="postgres",   # pg_ctl refuses to run as root
)
print(f"  pg_ctl: {out or err}")

# Poll for the promotion rather than sleeping a fixed number of seconds: a
# hard-coded sleep would be baked straight into the RTO we report below, and
# the notebook would be 'measuring' a constant.
new_primary = None
deadline = time.time() + 60
while time.time() < deadline:
    try:
        conn = psycopg2.connect(connect_timeout=2, **DB_STANDBY)
        conn.autocommit = True   # set BEFORE any query opens a transaction
        cur = conn.cursor()
        cur.execute("SELECT pg_is_in_recovery()")
        if not cur.fetchone()[0]:
            new_primary = conn
            break
        conn.close()
    except psycopg2.Error:
        pass
    time.sleep(0.1)

assert new_primary is not None, (
    "the standby never left recovery mode -- promotion failed, and the lab "
    "has no primary at all right now")
print("  ✅ Server left recovery mode — it is the new PRIMARY.")

# --------------------------------------------------------------- Step 5 ----
print("\nStep 5: VERIFY — does the new primary accept writes?")
cur = new_primary.cursor()
cur.execute(
    "INSERT INTO audit_log (table_name, record_id, action, changed_by) "
    "VALUES (%s, %s, %s, %s) RETURNING id",
    ('failover_test', 0, 'INSERT', 'failover_verification')
)
written_id = cur.fetchone()[0]
failover_time = time.time() - disaster_time
new_primary.close()
print(f"  ✅ Write #{written_id} succeeded on the new primary.")

print(f"\n🎉 FAILOVER COMPLETE in {failover_time:.1f} seconds")
print(f"   RTO (fence -> writable again): {failover_time:.1f} s")
print(f"   RPO (WAL lost at the fence):   {missing_bytes} bytes")
print("\n📌 Step 6 (failback) is NOT done. Port 5432 is dead, port 55433 is the")
print(f"   new primary, and write #{written_id} exists ONLY there. Notebook 3")
print("   and notebook 4 each open with a failback cell that repairs the")
print("   topology and carries that write back — run either one next.")
print("   Full reset instead: docker compose down -v && docker compose up -d")

assert failover_time < 300, (
    f"the failover took {failover_time:.0f}s -- something hung; this is not a "
    f"credible hot-standby RTO")


## 📝 Summary

### What You Learned

1. **WAL (Write-Ahead Log)** — Every database change is logged before being applied.
   Streaming replication sends these logs to the standby in real-time.

2. **Replication Lag** — The delay between primary and standby.
   Monitor it with `pg_stat_replication` and `pg_wal_lsn_diff()`.
   Lag is the floor on your RPO: with async replication you cannot promise a
   recovery point tighter than the lag you actually run at.

3. **Split-Brain** — When two servers both think they are primary.
   Prevent with fencing (STONITH), quorum, or lease-based systems.

4. **Manual Failover** — Record the WAL position, fence, verify the standby
   caught up, promote, verify writes. In production, automate this with
   Patroni or pg_auto_failover.

5. **RPO measured, not assumed** — Step 3 compared the primary's last WAL
   position with the standby's replay position. That difference *is* the data
   loss of this failover. Guessing it is how teams discover their real RPO
   during an incident instead of during a drill.

### Where the lab is now

Deliberately mid-disaster: **5432 is fenced and dead, 55433 is the new
primary**, and it holds one write that 5432 has never seen. That is not a
runnable state — and it is not a state you would leave production in either.
Notebooks 3 and 4 both begin with a **failback** cell that repairs it without
throwing that write away.

### Next Notebook

In **Notebook 3**, we explore backup strategies — full, incremental,
and differential backups, plus point-in-time recovery.
